# UniProtKB Programmatic Extraction: Engineering & Benchmarking

**Objective:** Evaluate and benchmark major programmatic methods for extracting sequence and metadata from UniProtKB to select the most robust strategy for an automated pipeline characterizing uncharacterized proteins.

**Official Documentation References**

- [UniProt API Interactive Swagger Docs](https://www.uniprot.org/api-documentation)
- [UniProt Programmatic Access Guide](https://www.uniprot.org/help/api)
- [UniProt Query Streams Help](https://www.uniprot.org/help/api_queries)


### 1. Importing libraries


In [41]:
# !pip install bioservices biopython pandas requests

In [42]:
# import necessary libraries
import requests
import pandas as pd
import time
import io
import json
from bioservices import UniProt
from Bio import ExPASy, SeqIO
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import uuid
from IPython.display import HTML, display

### 2. Configuration


In [43]:
# Configuration
PROTEOME_ID = 'UP000000579' # Haemophilus influenzae
SAMPLE_ACCESSIONS = ['P44101', 'P44249', 'P00567']
QUERY = f'(proteome:{PROTEOME_ID}) AND (protein_name:"uncharacterized")'

print("Environment ready. Target Proteome:", PROTEOME_ID)

Environment ready. Target Proteome: UP000000579


### 3. Error Handling & Rate Limit Resilience

Before implementing the methods, we define a robust requester with exponential backoff to handle `429` and `503` status codes.


In [44]:
def create_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS", "POST"]
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))
    return session

session = create_session()

### 4. Helper Function to view outputs properly


In [45]:
def show_json(data, max_depth=1):
    """
    Renders JSON in a VS Code style dark tree view with collapsible arrows.
    
    Parameters:
    - data: dict or str (JSON string or Python dictionary)
    - max_depth: how many levels deep to expand by default (e.g. 1 or 2)
    """
    if isinstance(data, str):
        data = json.loads(data)
        
    container_id = f"json_viewer_{uuid.uuid4().hex}"
    json_str = json.dumps(data)
    
    html_code = f"""
    <div id="{container_id}" style="background-color: #1e1e1e; padding: 14px; border-radius: 6px; font-family: 'Cascadia Code', 'Fira Code', Consolas, monospace; font-size: 13px; text-align: left; overflow-x: auto;"></div>
    
    <script src="https://cdn.jsdelivr.net/npm/renderjson@1.4.0/renderjson.min.js"></script>
    <script>
        (function() {{
            renderjson.set_show_to_level({max_depth});
            renderjson.set_icons('&#9654; ', '&#9660; ');
            
            var container = document.getElementById("{container_id}");
            if (container) {{
                container.appendChild(renderjson({json_str}));
            }}
        }})();
    </script>
    
    <style>
        #{container_id} .renderjson {{ color: #d4d4d4; line-height: 1.5; }}
        #{container_id} .renderjson .disclosure {{ color: #808080; cursor: pointer; text-decoration: none; font-weight: bold; margin-right: 2px; }}
        #{container_id} .renderjson .disclosure:hover {{ color: #ffffff; }}
        #{container_id} .renderjson .syntax {{ color: #ffd700; }}
        #{container_id} .renderjson .string {{ color: #ce9178; }}
        #{container_id} .renderjson .number {{ color: #b5cea8; }}
        #{container_id} .renderjson .boolean {{ color: #569cd6; }}
        #{container_id} .renderjson .key {{ color: #9cdcfe; }}
        #{container_id} .renderjson .keyword {{ color: #c586c0; }}
    </style>
    """
    display(HTML(html_code))

In [46]:
def show_fasta(fasta_text, max_records=10):
    """Renders a formatted FASTA text viewer with dark mode styling."""
    records = [r for r in fasta_text.strip().split('>') if r]
    
    html_code = f"""
    <div style="background-color: #1e1e1e; padding: 14px; border-radius: 6px; 
                font-family: 'Cascadia Code', 'Fira Code', Consolas, monospace; 
                font-size: 12px; max-height: 400px; overflow-y: auto; text-align: left;">
    """
    
    for rec in records[:max_records]:
        lines = rec.strip().split('\n')
        header = lines[0]
        seq = "".join(lines[1:])
        
        html_code += f"""
        <div style="margin-bottom: 12px; border-bottom: 1px solid #333333; padding-bottom: 8px;">
            <div style="color: #569cd6; font-weight: bold; word-break: break-all;">&gt;{header}</div>
            <div style="color: #ce9178; word-break: break-all; margin-top: 4px; line-height: 1.4;">{seq}</div>
        </div>
        """
        
    html_code += f"""
        <div style="color: #808080; font-style: italic; margin-top: 8px;">
            Showing first {min(max_records, len(records))} of {len(records)} total sequences...
        </div>
    </div>
    """
    display(HTML(html_code))

# Display preview of first 10 FASTA records
show_fasta(fasta_data, max_records=10)

## Method 1: Single Entry REST Endpoint

Fetches specific entries. Useful for targeted updates but suffers from high overhead for large batches.


In [47]:
def get_single_entry(accession, format='json'):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.{format}"
    response = session.get(url)
    response.raise_for_status()
    return response.text

# Demo
# print("Method 1 (JSON Sample):", get_single_entry('P44101')[:200], "...")

In [48]:
import json

# Parse JSON response and display key fields
raw_json = get_single_entry('P44101')
parsed_json = json.loads(raw_json)

print("=== Method 1 Output Inspection ===")
print(f"Accession:      {parsed_json.get('primaryAccession')}")
print(f"Entry Type:     {parsed_json.get('entryType')}")
print(f"Organism:       {parsed_json.get('organism', {}).get('scientificName')}")
print(f"Protein Name:   {parsed_json.get('proteinDescription', {}).get('recommendedName', {}).get('fullName', {}).get('value')}")
print(f"Sequence Length: {parsed_json.get('sequence', {}).get('length')} aa")

# Opens top 1 levels by default, leaving deeper dictionaries collapsed
show_json(raw_json)

=== Method 1 Output Inspection ===
Accession:      P44101
Entry Type:     UniProtKB reviewed (Swiss-Prot)
Organism:       Haemophilus influenzae (strain ATCC 51907 / DSM 11121 / KW20 / Rd)
Protein Name:   Uncharacterized protein HI_1043
Sequence Length: 166 aa


## Method 2: Search & Query Stream API

Best for discovery based on metadata (e.g., 'uncharacterized'). `stream` is preferred over `search` for large results to avoid pagination overhead.


In [49]:
def stream_query(query, fields="accession,reviewed,length,sequence"):
    url = "https://rest.uniprot.org/uniprotkb/stream"
    params = {'query': query, 'format': 'tsv', 'fields': fields}
    response = session.get(url, params=params)
    if response.status_code != 200:
        print(f"Error: {response.text}")
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(response.text), sep='\t')

# Demo
df_stream = stream_query(QUERY)
print(f"Method 2: Found {len(df_stream)} entries.")
# display(df_stream.head(3))

Method 2: Found 358 entries.


In [50]:
# Inspect the resulting DataFrame
print("=== Method 2 Output Inspection ===")
print(f"DataFrame Shape: {df_stream.shape} (Rows, Columns)")
print("\nColumns:", list(df_stream.columns))
n = 3
print(f"\nFirst {n} records:")
display(df_stream.head(n))

=== Method 2 Output Inspection ===
DataFrame Shape: (358, 4) (Rows, Columns)

Columns: ['Entry', 'Reviewed', 'Length', 'Sequence']

First 3 records:


,Entry,Reviewed,Length,Sequence
0,P44693,reviewed,475,MPVQHVKLARDRRKKRAYIKVGVFFVAILLILTGILLTIKSKPEEN...
1,O05031,reviewed,265,MTGAQLIMACLKAHHVTTLFGYPGGAIMPTYDALYDAGLDHLLCRN...
2,O05083,reviewed,353,MKKIGFFIMNIGSAGGTERVSINVANALAKQGYDVSFISIGGNKPF...


## Method 3: Asynchronous Batch / ID Mapping API

The enterprise choice for large, known lists of accessions. Involves submission, polling, and retrieval.


In [51]:
def batch_id_mapping(accessions):
    """
    Method 3: Asynchronous Batch / ID Mapping API
    Updated: Checks for 'results' key in status response for completion.
    """
    # 1. Submit the job
    sub_url = "https://rest.uniprot.org/idmapping/run"
    payload = {'from': 'UniProtKB_AC-ID', 'to': 'UniProtKB', 'ids': ','.join(accessions)}
    req = session.post(sub_url, data=payload)
    req.raise_for_status()
    job_id = req.json()['jobId']

    # 2. Poll for completion
    print(f"Job Submitted. ID: {job_id}")
    max_retries = 30
    for i in range(max_retries):
        status_res = session.get(f"https://rest.uniprot.org/idmapping/status/{job_id}")
        status_res.raise_for_status()
        status_data = status_res.json()

        # If 'results' is present, the job is finished and results are here
        if 'results' in status_data:
            return [item['to'] for item in status_data['results'] if 'to' in item]

        # Otherwise, check for explicit failure in the status key if it exists
        if status_data.get('jobStatus') in ['FAILED', 'ERROR']:
            raise Exception(f"Job failed: {status_data}")

        if i % 2 == 0:
            print(f"Polling {job_id}... Status: {status_data.get('jobStatus', 'RUNNING')}")
        time.sleep(2)

    raise TimeoutError("ID mapping job timed out.")

# Demo
try:
    batch_res = batch_id_mapping(SAMPLE_ACCESSIONS)
    print(f"Method 3: Successfully retrieved {len(batch_res)} entries.")
except Exception as e:
    print(f"Method 3 execution error: {e}")

Job Submitted. ID: hdASP3VVwK
Method 3: Successfully retrieved 3 entries.


In [52]:
# Inspect mapped accessions list
print("=== Method 3 Output Inspection ===")
print(f"Return Type:        {type(batch_res)}")
print(f"Mapped Accessions:  {batch_res}")

show_json(batch_res)

=== Method 3 Output Inspection ===
Return Type:        <class 'list'>
Mapped Accessions:  [{'entryType': 'UniProtKB reviewed (Swiss-Prot)', 'primaryAccession': 'P44101', 'uniProtkbId': 'Y1043_HAEIN', 'entryAudit': {'firstPublicDate': '1995-11-01', 'lastAnnotationUpdateDate': '2026-01-28', 'lastSequenceUpdateDate': '1995-11-01', 'entryVersion': 124, 'sequenceVersion': 1}, 'annotationScore': 2.0, 'organism': {'scientificName': 'Haemophilus influenzae (strain ATCC 51907 / DSM 11121 / KW20 / Rd)', 'taxonId': 71421, 'lineage': ['Bacteria', 'Pseudomonadati', 'Pseudomonadota', 'Gammaproteobacteria', 'Pasteurellales', 'Pasteurellaceae', 'Haemophilus']}, 'proteinExistence': '4: Predicted', 'proteinDescription': {'recommendedName': {'fullName': {'value': 'Uncharacterized protein HI_1043'}}}, 'genes': [{'orderedLocusNames': [{'value': 'HI_1043'}]}], 'features': [{'type': 'Chain', 'location': {'start': {'value': 1, 'modifier': 'EXACT'}, 'end': {'value': 166, 'modifier': 'EXACT'}}, 'description': '

## Method 4: Direct Proteome Stream

Directly fetches a full proteome. Highly optimized for bulk downloads.


In [53]:
def get_proteome(proteome_id, format='fasta'):
    url = f"https://rest.uniprot.org/uniprotkb/stream?query=proteome:{proteome_id}&format={format}"
    response = session.get(url)
    response.raise_for_status()
    return response.text

# Demo
fasta_data = get_proteome(PROTEOME_ID)
print(f"Method 4: Downloaded {len(fasta_data.split('>'))-1} FASTA sequences.")

Method 4: Downloaded 1703 FASTA sequences.


In [54]:
# Inspect raw FASTA stream
print("=== Method 4 Output Inspection ===")
print(f"Total Raw Output Size: {len(fasta_data):,} characters")

# Split into FASTA records
fasta_records = [rec for rec in fasta_data.strip().split('>') if rec]
print(f"Total FASTA Records:   {len(fasta_records)}")

print("\n--- Preview of FASTA Records ---")
show_fasta(fasta_data, max_records=10)

=== Method 4 Output Inspection ===
Total Raw Output Size: 803,426 characters
Total FASTA Records:   1703

--- Preview of FASTA Records ---


## Method 5: High-Level Python Wrapper

Using `bioservices` for abstracted access.


In [55]:
def bioservices_lookup(accessions):
    u = UniProt()
    # bioservices retrieve uses comma separated strings or lists
    return u.retrieve(accessions)

# Demo
lib_res = bioservices_lookup(SAMPLE_ACCESSIONS)
print(f"Method 5: Library wrapper successfully retrieved entries.")

Method 5: Library wrapper successfully retrieved entries.


In [56]:
# Inspect BioServices retrieval output
print("=== Method 5 Output Inspection ===")
print(f"Return Type: {type(lib_res)}")
print("\n--- Raw Response Preview ---")
# print(str(lib_res)[:350] + "\n...")
show_json(lib_res)

=== Method 5 Output Inspection ===
Return Type: <class 'list'>

--- Raw Response Preview ---


## Benchmarking & Comparison


In [57]:
import time
benchmark_results = []

# Defining the test methods with unified return types for length counting
methods = [
    ("Single REST (GET)", lambda: [get_single_entry(acc) for acc in SAMPLE_ACCESSIONS]),
    ("Stream API (Query)", lambda: stream_query(QUERY)),
    ("Async Batch (Mapping)", lambda: batch_id_mapping(SAMPLE_ACCESSIONS)),
    ("Proteome Stream", lambda: get_proteome(PROTEOME_ID).split('>') [1:]),
    ("BioServices Wrapper", lambda: bioservices_lookup(SAMPLE_ACCESSIONS))
]

print("Starting Robust Benchmark Execution...")
for name, func in methods:
    print(f"Running {name}...")
    start = time.time()
    try:
        result = func()
        duration = round(time.time() - start, 3)

        # Count items based on return type (DataFrame, List, or Raw Text processed to list)
        if isinstance(result, pd.DataFrame):
            count = len(result)
        elif isinstance(result, list):
            count = len(result)
        else:
            count = "N/A"

        print(f"  -> Success! Time: {duration}s | Items: {count}")
        benchmark_results.append({"Method": name, "Time (s)": duration, "Count": count})
    except Exception as e:
        print(f"  -> Method {name} failed: {e}")
        benchmark_results.append({"Method": name, "Time (s)": "Error", "Count": 0})

# Display Results
df_bench = pd.DataFrame(benchmark_results)
display(df_bench)

Starting Robust Benchmark Execution...
Running Single REST (GET)...
  -> Success! Time: 0.586s | Items: 3
Running Stream API (Query)...
  -> Success! Time: 0.216s | Items: 358
Running Async Batch (Mapping)...
Job Submitted. ID: hdASP3VVwK
  -> Success! Time: 0.598s | Items: 3
Running Proteome Stream...
  -> Success! Time: 0.532s | Items: 1703
Running BioServices Wrapper...
  -> Success! Time: 3.315s | Items: 3


,Method,Time (s),Count
0,Single REST (GET),0.586,3
1,Stream API (Query),0.216,358
2,Async Batch (Mapping),0.598,3
3,Proteome Stream,0.532,1703
4,BioServices Wrapper,3.315,3


### Final Comparison Table

| Method              | Scalability | Maintenance | Field Selection | Best For                     |
| :------------------ | :---------- | :---------- | :-------------- | :--------------------------- |
| **Single REST**     | Low         | High        | Medium          | Single lookups               |
| **Stream API**      | High        | Low         | High            | Metadata-based discovery     |
| **Async Batch**     | Very High   | Medium      | High            | Large static accession lists |
| **Proteome Stream** | Maximum     | Low         | Low             | Full proteome downloads      |
| **Wrappers**        | Medium      | Very Low    | Low             | Rapid prototyping            |

### Final Recommendation

For a **zero-maintenance, automated pipeline** processing hundreds of proteins, I recommend **Method 2 (Stream API)** if the list is dynamic or query-based, or **Method 3 (Async Batch)** if you have a static list of accessions.

**Method 2** offers the best balance of speed, payload customization (getting only what you need), and code simplicity without the polling logic required by the asynchronous API.


## Implementation of 5 Programmatic Methods

The notebook implements five distinct ways to retrieve data from UniProtKB:

### **Method 1: Single Entry REST Endpoint**

- **Mechanism:** Direct `GET` requests to fetch individual accession entries in JSON format.
- **Best For:** Quick single-protein lookups; inefficient for bulk retrieval.

---

### **Method 2: Search & Query Stream API**

- **Mechanism:** Uses UniProt's `/stream` endpoint to filter and stream results directly into a `pandas` DataFrame (e.g., retrieving uncharacterized proteins).
- **Best For:** Metadata-based filtering without the overhead of pagination.

---

### **Method 3: Asynchronous Batch / ID Mapping API**

- **Mechanism:** Submits a batch job ID, polls UniProt until processing completes, and retrieves mapped accessions.
- **Best For:** Large, static lists of known accession numbers.

---

### **Method 4: Direct Proteome Stream**

- **Mechanism:** Streams complete proteome files directly in FASTA format.
- **Best For:** Downloading whole-genome protein sequence sets at once.

---

### **Method 5: High-Level Python Wrapper (`bioservices`)**

- **Mechanism:** Uses the `bioservices.UniProt` library wrapper to fetch protein details.
- **Best For:** Rapid prototyping with minimal boilerplate code.
